

# Part 4 — Spark Structured Streaming for Synthetic Transactions
    This component uses generated JSON events to demonstrate incremental processing, event-time windows, checkpointing, and illustrative alerts. It does not detect real fraud.

In [ ]:
# Generate events from a terminal:
    # python scripts/04_generate_transactions.py --output streaming/incoming --count 1000 --batch-size 100
    # For Colab, make sure the generated JSON files are available in the runtime directory.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
    spark = SparkSession.builder.appName("BankingStreaming").getOrCreate()
    schema = T.StructType([
        T.StructField("transaction_id",T.StringType()),
        T.StructField("account_id",T.StringType()),
        T.StructField("event_time",T.StringType()),
        T.StructField("amount",T.DoubleType()),
        T.StructField("merchant_category",T.StringType()),
        T.StructField("channel",T.StringType()),
        T.StructField("country",T.StringType())
    ])
    input_dir = "streaming/incoming"
    events = (spark.readStream.schema(schema).option("maxFilesPerTrigger",1).json(input_dir)
              .withColumn("event_ts",F.to_timestamp("event_time"))
              .withWatermark("event_ts","2 minutes"))

In [ ]:
flagged = events.withColumn(
        "alert_reason",
        F.when(F.col("amount") >= 2500, "HIGH_AMOUNT")
         .when((F.col("country") != "IN") & (F.col("amount") >= 500), "FOREIGN_HIGH_AMOUNT")
    )
    alerts = flagged.filter(F.col("alert_reason").isNotNull())
    alert_query = (alerts.writeStream.format("console").outputMode("append")
                   .option("truncate","false")
                   .option("checkpointLocation","outputs/checkpoints/alerts")
                   .start())

In [ ]:
window_summary = (events.groupBy(F.window("event_ts","1 minute"),"channel")
                     .agg(F.count("*").alias("transaction_count"),
                          F.round(F.sum("amount"),2).alias("total_amount")))
    summary_query = (window_summary.writeStream.format("console").outputMode("append")
                     .option("checkpointLocation","outputs/checkpoints/summary").start())
    # Keep the streaming session alive while adding new JSON files to the input directory.
    # Stop the queries with alert_query.stop(); summary_query.stop(); spark.stop()

## Concepts and limitations
    - The file source simulates arrivals as new files appear.
    - Checkpoints support recovery and record progress.
    - Watermarks bound state and define how late event-time data is handled.
    - A production system might use Kafka or Azure Event Hubs and a durable alert sink.
    - Thresholds are illustrative and will create false positives; they are not a validated fraud model.